# Screening cuantitativo de acciones y ETFs

Corre el modelo completo del repo sobre datos que se bajan en vivo de Yahoo Finance. Menú → **Entorno de ejecución → Ejecutar todo**.

## Independiente de tu portafolio

Este notebook **no lee ninguna cuenta**. Cada nombre se puntúa por sus propios méritos: el bloque *Portfolio Fit* está removido del modelo, no puesto en cero.

La distinción importa. Pasar un libro vacío no habría bastado: con cero posiciones, `existing_overlap` sigue devolviendo `0.0` para cada nombre — un número real, idéntico en todos — que el motor estandarizaría y contaría como bloque poblado. Una cuenta vacía seguiría influyendo en el compuesto. Quitar el bloque es la única forma de que el screen sea de verdad independiente.

## Perfil de riesgo

Eliges **Conservador Defensivo**, **Conservador**, **Moderado** o **Agresivo** en Parámetros, y eso reconfigura cuatro cosas a la vez — no es una etiqueta sobre el mismo ranking:

1. **Pesos de los bloques** — qué premia el score compuesto.
2. **Umbrales de recomendación** — cuánto score exige un Overweight y qué tan poco basta para un Underweight. Asimétricos a propósito.
3. **Gates de riesgo** — los techos duros que solo pueden degradar una recomendación.
4. **Dimensionamiento y elegibilidad** — volatilidad objetivo, tope por posición y liquidez mínima para siquiera entrar al ranking.

## Qué cambia al usar Yahoo en vez de IBKR

| | IBKR | Yahoo |
|---|---|---|
| Precio, máx/mín 52s, volumen, dividendos | ✅ | ✅ |
| Universo | 21 nombres del snapshot | 447 candidatos, o el que definas |
| Datos | congelados en la captura | en vivo |
| Vol implícita (`iv_hv_spread`) | ✅ | opcional, lento |
| Percentil de IV a 52s (`iv_percentile`) | ✅ | **no existe** |

Yahoo publica la cadena de opciones de hoy, no un histórico de volatilidad implícita, así que el percentil de IV no se puede reconstruir. Esa métrica se **omite**, no se rellena con cero: el motor renormaliza los pesos del bloque sobre las métricas que sí están. La celda de cobertura te muestra exactamente cuánto pesa esa ausencia antes de que mires un solo ranking.


## 1 · Instalación y motor


In [ ]:
%pip install -q yfinance openpyxl cvxpy scikit-learn
print('yfinance listo')


In [ ]:
# El paquete screener/ del repo, embebido. Se extrae a /content.
import base64, gzip, hashlib, io, sys, tarfile

ENGINE_SHA256 = "fb3ce565c6d877b3a88ae303656c62356c513555867489d63cefbf2b97531f76"
ENGINE_B64 = (
    "..."
)

raw = gzip.decompress(base64.b64decode(ENGINE_B64))
digest = hashlib.sha256(raw).hexdigest()
assert digest == ENGINE_SHA256, f'engine checksum mismatch: {digest}'

with tarfile.open(fileobj=io.BytesIO(raw)) as tar:
    try:
        tar.extractall('.', filter='data')  # Python 3.12+
    except TypeError:
        tar.extractall('.')
if '.' not in sys.path:
    sys.path.insert(0, '.')

import screener
from screener.profiles import PROFILES

# Deliberadamente NO se importa FACTOR_MODEL aqui. El perfil lo
# reemplaza mas abajo, y un nombre enlazado ahora quedaria obsoleto:
# seguiria apuntando al modelo de 7 bloques con Portfolio Fit incluido.
print(f'motor verificado  sha256={digest[:16]}...')
print(f'perfiles disponibles: {", ".join(p.label for p in PROFILES.values())}')


In [ ]:
# Ayudas de presentacion. Mismo par divergente que la pagina HTML del
# repo, validado para daltonismo: naranja = adverso, arena = neutro,
# azul = favorable. Sin matplotlib, y eligiendo el color del texto por
# luminancia — background_gradient de pandas deja texto negro sobre
# azul oscuro, que es ilegible.
import numpy as np
import pandas as pd

_NARANJA, _NEUTRO, _AZUL = (194, 65, 12), (232, 228, 222), (3, 105, 161)

def _mezcla(a, b, t):
    return tuple(round(x + (y - x) * t) for x, y in zip(a, b))

def escala(v, vmin=-2.0, vmax=2.0):
    """Estilo CSS para un valor, divergente alrededor del punto medio."""
    if v is None or (isinstance(v, float) and not np.isfinite(v)):
        return ''
    t = min(1.0, max(0.0, (float(v) - vmin) / (vmax - vmin)))
    rgb = (_mezcla(_NARANJA, _NEUTRO, t * 2) if t < 0.5
           else _mezcla(_NEUTRO, _AZUL, (t - 0.5) * 2))
    luma = 0.2126 * rgb[0] + 0.7152 * rgb[1] + 0.0722 * rgb[2]
    return f"background-color:rgb{rgb};color:{'#1C1917' if luma > 140 else '#FFFFFF'}"


## 2 · Parámetros

`Universo completo` son 447 candidatos (317 acciones del S&P + Nasdaq-100 + Dow, sin duplicar, y 130 ETFs curados) y tarda 1-3 min en bajar. De ahí, la política de selección decide cuáles se evalúan.


In [ ]:
# @markdown ### Universo y ventana
UNIVERSO = "Completo (S&P + Nasdaq + Dow + ETFs)"  # @param ["Completo (S&P + Nasdaq + Dow + ETFs)", "Solo acciones (S&P + Nasdaq + Dow)", "Solo ETFs", "Solo Nasdaq-100", "Solo Dow 30", "Lista personalizada"]
TICKERS_PERSONALIZADOS = ""  # @param {type:"string"}
# @markdown Separados por coma. Solo aplica si elegiste "Lista personalizada".

BENCHMARK = "SPY"  # @param {type:"string"}
PERIODO = "2y"  # @param ["1y", "2y", "5y"]
TASA_LIBRE_RIESGO = 0.0425  # @param {type:"number"}

# @markdown ### Perfil de riesgo
PERFIL = "Moderado"  # @param ["Conservador Defensivo", "Conservador", "Moderado", "Agresivo"]
# @markdown Cambia pesos de bloque, umbrales de recomendación, gates de riesgo, dimensionamiento y liquidez mínima — todo a la vez.
TAMANO_POSICION_USD = 500000  # @param {type:"number"}
# @markdown Tamaño de posición que asume el bloque de liquidez para calcular `days_to_liquidate`. Es un supuesto de dimensionamiento, no un dato de tu cuenta.

# @markdown ### Datos opcionales (lentos)
CON_VOL_IMPLICITA = False  # @param {type:"boolean"}
# @markdown Baja la cadena de opciones para `iv_hv_spread`. ~2 requests por ticker.
CON_NOMBRES_Y_SECTORES = False  # @param {type:"boolean"}
# @markdown Necesario si usas lista personalizada: sin el nombre largo, el filtro de productos apalancados/inversos no puede actuar.

from screener.yahoo_adapter import default_universe

_GRUPOS = {
    "Completo (S&P + Nasdaq + Dow + ETFs)": ("SP500", "NDX", "DJIA", "ETF"),
    "Solo acciones (S&P + Nasdaq + Dow)": ("SP500", "NDX", "DJIA"),
    "Solo ETFs": ("ETF",),
    "Solo Nasdaq-100": ("NDX",),
    "Solo Dow 30": ("DJIA",),
}

if UNIVERSO == "Lista personalizada":
    TICKERS = [t.strip().upper().replace('.', '-')
               for t in TICKERS_PERSONALIZADOS.split(',') if t.strip()]
    if not TICKERS:
        raise ValueError('Elegiste lista personalizada pero no pusiste tickers.')
    if BENCHMARK.upper() not in TICKERS:
        TICKERS.append(BENCHMARK.upper())
    if not CON_NOMBRES_Y_SECTORES:
        print('AVISO: sin nombres largos, un ETF apalancado o de covered-call\n'
              '       en tu lista pasaria el filtro de producto. Considera\n'
              '       activar CON_NOMBRES_Y_SECTORES.')
else:
    TICKERS = default_universe(_GRUPOS[UNIVERSO], benchmark=BENCHMARK)

print(f'{len(TICKERS)} tickers  |  benchmark {BENCHMARK}  |  {PERIODO} de historia diaria')

from screener.profiles import get_profile

perfil = get_profile(PERFIL)
print()
print(perfil.describe())


## 3 · Bajar datos


In [ ]:
import time
from screener.yahoo_adapter import fetch_market_data

_t0 = time.time()
market_data, frame_diario = fetch_market_data(
    TICKERS,
    benchmark=BENCHMARK,
    risk_free_rate=TASA_LIBRE_RIESGO,
    period=PERIODO,
    with_metadata=CON_NOMBRES_Y_SECTORES,
    with_iv=CON_VOL_IMPLICITA,
    progress=True,
    with_frame=True,   # el optimizador necesita retornos diarios
)

print(f'\n{len(market_data["instruments"])} instrumentos utilizables en {time.time() - _t0:.0f}s')

_dropped = market_data.get('dropped', [])
if _dropped:
    print(f'\n{len(_dropped)} descartados antes de puntuar:')
    for _t, _r in _dropped[:15]:
        print(f'  {_t:8s} {_r}')
    if len(_dropped) > 15:
        print(f'  ... y {len(_dropped) - 15} mas')


## 4 · Cobertura de métricas

Léela antes del ranking. Una métrica con cobertura baja se está estandarizando contra una sección transversal chica mientras el resto del universo se puntúa sin ella.


In [ ]:
from screener.yahoo_adapter import coverage_report

_cov = coverage_report(market_data)
_faltantes = _cov[_cov['coverage'] < 1.0]

if _faltantes.empty:
    print('Cobertura completa en las 28 metricas.')
else:
    print('Metricas por debajo de cobertura total:\n')
    for _, _r in _faltantes.iterrows():
        print(f"  {_r['coverage']:6.1%}  {_r['metric']:34s} ({_r['block']}) — {_r['source']}")

(_cov.style
    .format({'coverage': '{:.0%}'})
    .map(lambda v: escala(v, 0.0, 1.0), subset=['coverage'])
    .hide(axis='index'))


## 5 · Correr el modelo


In [ ]:
from screener.run_screen import run_standalone
from screener.report import console_summary
from screener.seleccion import (CRITERIOS, politica_declarada,
                               tabla as tabla_seleccion)

# Sin libro: ninguna cuenta se lee y el bloque Portfolio Fit no esta
# en el modelo. El perfil reconfigura pesos, umbrales, gates,
# dimensionamiento y elegibilidad de una sola vez.
scored, meta = run_standalone(
    market_data,
    profile=PERFIL,
    position_usd=TAMANO_POSICION_USD,
    rf=TASA_LIBRE_RIESGO,
)
print(console_summary(scored, meta))

# Politica de seleccion del universo: quien entro, quien no, y por que.
print()
print(politica_declarada())
_sel = meta['seleccion_resumen']
print(f"\nCandidatos: {_sel['candidatos']}  ->  admitidos: {_sel['admitidos']}")
for _c in CRITERIOS:
    if _sel.get(_c.clave):
        print(f'  rechazados por {_c.titulo.lower()}: {_sel[_c.clave]}')
universo = tabla_seleccion(meta['seleccion'])
_fuera = universo[universo['admitido'] == 'no']
if not _fuera.empty:
    print()
    for _r in _fuera.head(25).itertuples():
        print(f'  {_r.ticker:8s} [{_r.criterio}] {_r.motivo[:66]}')


## 6 · Ranking

`indicative_weight` es tamaño por volatilidad inversa escalado por convicción, con topes duros — un punto de partida para dimensionar, no una orden.


In [ ]:
import screener.config as _cfg

# El modelo VIGENTE, ya con el perfil aplicado: seis bloques, sin
# Portfolio Fit. Se lee aqui y no al importar, por la misma razon.
MODELO = _cfg.FACTOR_MODEL
BLOQUES = [b.key for b in MODELO]

tabla = pd.DataFrame([{
    'rank': i,
    'ticker': r.ticker,
    'tipo': r.asset_type,
    'reco': r.recommendation,
    'score': r.score_0_100,
    'z': r.composite_z,
    'peso_ind': r.indicative_weight,
    'ret_1a': r.diagnostics.get('return_1y'),
    'vol': r.diagnostics.get('volatility'),
    'max_dd': r.diagnostics.get('max_drawdown'),
    'beta': r.diagnostics.get('beta'),
    'sharpe': r.raw_metrics.get('sharpe_1y'),
    # Sin libro no hay correlacion contra el libro. Se muestra alfa
    # anualizado en su lugar, no una columna vacia.
    'alpha': r.diagnostics.get('alpha_annual'),
    'gates': ', '.join(r.gates_triggered),
} for i, r in enumerate(scored, 1)])

PORCENTAJES = ['peso_ind', 'ret_1a', 'vol', 'max_dd', 'alpha']

def pintar_reco(v):
    return {
        'OVERWEIGHT': 'background-color:#0369A1;color:white;font-weight:600',
        'UNDERWEIGHT': 'background-color:#C2410C;color:white;font-weight:600',
    }.get(v, 'color:#57534E')

(tabla.head(40).style
    .format({c: '{:.1%}' for c in PORCENTAJES} |
            {'score': '{:.1f}', 'z': '{:+.2f}', 'beta': '{:.2f}',
             'sharpe': '{:.2f}'}, na_rep='—')
    .map(pintar_reco, subset=['reco'])
    .map(lambda v: escala(v, 20, 80), subset=['score'])
    .hide(axis='index'))


## 7 · Mapa de factores

Dónde gana o pierde cada nombre. Un score compuesto alto sostenido por un solo bloque es frágil de una forma que el ranking no te muestra.


In [ ]:
ETIQUETAS = {b.key: b.label for b in MODELO}

mapa = pd.DataFrame(
    [{'ticker': r.ticker, **{ETIQUETAS[k]: r.block_scores.get(k)
                             for k in BLOQUES}}
     for r in scored[:30]]
).set_index('ticker')

(mapa.style
    .format('{:+.2f}', na_rep='—')
    .map(escala)
    .set_caption('Score z por bloque — azul favorable, naranja adverso'))


## 8 · Detalle de un nombre


In [ ]:
TICKER = "NVDA"  # @param {type:"string"}

from screener.config import all_metrics

_r = next((r for r in scored if r.ticker == TICKER.upper()), None)
if _r is None:
    _excluidos = dict(meta.get('excluded', []))
    if TICKER.upper() in _excluidos:
        print(f'{TICKER.upper()} fue excluido por filtros duros:')
        for _m in _excluidos[TICKER.upper()]:
            print(f'  - {_m}')
    else:
        print(f'{TICKER.upper()} no esta en el universo corrido.')
else:
    print(f'{_r.ticker} — {_r.name}')
    print(f'{_r.recommendation}   score {_r.score_0_100:.1f}/100   z {_r.composite_z:+.2f}   peso indicativo {_r.indicative_weight:.2%}')
    if _r.pre_gate_recommendation != _r.recommendation:
        print(f'\nDegradado desde {_r.pre_gate_recommendation} por:')
        for _g in _r.gates_triggered:
            print(f'  - {_g}')
    if _r.duplicates:
        print(f"\nExposicion duplicada: {', '.join(_r.duplicates)}")

    print('\nBloques')
    for _b in MODELO:
        _s = _r.block_scores.get(_b.key)
        _c = _r.block_coverage.get(_b.key, 0.0)
        _bar = '#' * int(max(0, min(4, (_s or 0) + 2)) * 5)
        print(f'  {_b.label:34s} {_s:+.2f}  cob {_c:4.0%}  {_bar}'
              if _s is not None else f'  {_b.label:34s}    —')

    print('\nMetricas crudas')
    _defs = all_metrics()
    for _k, _v in _r.raw_metrics.items():
        if _v is None or _k not in _defs:
            continue
        print(f'  {_defs[_k].label:36s} {_v:12.4f}   z {_r.metric_z.get(_k, float("nan")):+.2f}')


## 9 · Comparar los perfiles

El mismo universo, los mismos datos, cuatro configuraciones. Un nombre que aparece Overweight en todas es una señal robusta; uno que solo sobrevive en Agresivo te está diciendo que su score depende de que le perdones la volatilidad.

**`n/e` no es un error.** Cada perfil tiene su propio piso de liquidez ($100MM / $50MM / $20MM / $10MM de volumen diario), así que un nombre puede ser elegible para uno y no para otro. Cuando eso pasa, el más estricto lo marca como no elegible y te dice por qué.


In [ ]:
from screener.profiles import PROFILES
from screener.tuning import reset_all

_recos, _excluidos = {}, {}
try:
    for _k, _p in PROFILES.items():
        _s, _m = run_standalone(market_data, profile=_k,
                                position_usd=TAMANO_POSICION_USD,
                                rf=TASA_LIBRE_RIESGO)
        _recos[_p.label] = {r.ticker: r.recommendation for r in _s}
        _excluidos[_p.label] = dict(_m.get('excluded', []))
finally:
    # Deja el modelo como lo espera el resto del notebook.
    reset_all()
    scored, meta = run_standalone(market_data, profile=PERFIL,
                                  position_usd=TAMANO_POSICION_USD,
                                  rf=TASA_LIBRE_RIESGO)

NO_ELEGIBLE = 'NO ELEGIBLE'
_tickers = [r.ticker for r in scored]

# Cada perfil filtra por liquidez distinto, asi que no todos puntuan
# el mismo conjunto de nombres. Indexar a ciegas aqui reventaria con
# KeyError en cuanto un perfil excluya algo que otro si acepto.
comparacion = pd.DataFrame({
    _label: pd.Series({t: _r.get(t, NO_ELEGIBLE) for t in _tickers})
    for _label, _r in _recos.items()
})
comparacion.insert(0, 'score_' + PERFIL.lower(),
                   pd.Series({r.ticker: r.score_0_100 for r in scored}))

_ABREV = {'OVERWEIGHT': 'OW', 'MARKET WEIGHT': 'MW',
          'UNDERWEIGHT': 'UW', NO_ELEGIBLE: 'n/e'}
_TONO = {'OVERWEIGHT': 1.6, 'MARKET WEIGHT': 0.0, 'UNDERWEIGHT': -1.6}
_PERFILES = [p.label for p in PROFILES.values()]

def _estilo_reco(v):
    # No elegible es una categoria aparte, no un punto de la escala.
    if v == NO_ELEGIBLE:
        return 'background-color:#F5F5F4;color:#A8A29E;font-style:italic'
    return escala(_TONO.get(v, 0.0))

_ow = comparacion[_PERFILES].eq('OVERWEIGHT').sum(axis=1)
print(f'Overweight en TODOS los perfiles: {list(comparacion.index[_ow == len(_PERFILES)]) or "ninguno"}')
print(f'Overweight solo en Agresivo:     '
      f'{list(comparacion.index[(_ow == 1) & comparacion["Agresivo"].eq("OVERWEIGHT")]) or "ninguno"}')

for _label in _PERFILES:
    _fuera = [t for t in _tickers if _recos[_label].get(t) is None]
    if _fuera:
        print(f'\n{_label} no considera {len(_fuera)} de estos nombres:')
        for _t in _fuera[:8]:
            _razon = (_excluidos[_label].get(_t) or ['fuera del universo'])[0]
            print(f'  {_t:8s} {_razon}')

(comparacion.head(30).style
    .format({comparacion.columns[0]: '{:.1f}'})
    .format(lambda v: _ABREV.get(v, v), subset=_PERFILES)
    .map(_estilo_reco, subset=_PERFILES)
    .map(lambda v: escala(v, 20, 80), subset=[comparacion.columns[0]])
    .set_caption('Recomendación por perfil'))


## 10 · Views para Black-Litterman (CCI)

Los dos sistemas son complementarios y la frontera es nítida: **el screener decide sobre qué nombres hay una view y cuán fuerte es; Black-Litterman decide los pesos.**

Esta celda exporta los insumos tácticos — `Q` y convicción — en el esquema exacto que ya consumen `flujo_aprobacion` y `black_litterman_core` de tu notebook de CCI. No exporta pesos: bajo Black-Litterman los pesos salen del optimizador sujeto al Procedimiento de Inversión, y mandar un segundo juego de pesos sin restricciones al lado invita justo la confusión que una revisión de riesgo model existe para evitar.

### Cómo se traduce un ranking a un retorno esperado

Un z-score transversal es un **ranking**, no un pronóstico. La conversión es explícita:

$$Q_i = IC \times z_i \times \sigma_i$$

Escalado por riesgo (a igual ranking, el nombre más volátil merece mayor retorno esperado, que es lo que el optimizador media-varianza necesita para dimensionar bien) y centrado (un nombre en el medio de la sección transversal da exactamente cero).

**El IC es un supuesto declarado, no una estimación.** Es la correlación asumida entre el ranking del screener y los retornos realizados. El 0.08 por defecto es deliberadamente modesto y produce views dentro de la banda ±5% de tu documento técnico. No está calibrado contra ningún backtest.

La convicción es otra cosa: alimenta Ω y mide **confianza en la estimación** — cuántos de los seis bloques coinciden en signo, cuánta cobertura de datos hubo, si se activó un gate. Un nombre en z=+1.5 sostenido por un solo bloque no merece la misma Ω que uno donde los seis coinciden.


In [ ]:
from screener.black_litterman import (ViewParams, build_basket,
                                      build_views, public_view,
                                      write_views)
from screener.profiles import CCI_STRATEGIES, profile_for_strategy

# @markdown Estrategia de destino en el sistema BL de CCI.
ESTRATEGIA_CCI = "Moderado"  # @param ["Conservador_Defensivo", "Conservador", "Moderado", "Agresivo"]
IC_SUPUESTO = 0.08  # @param {type:"number"}
MAX_VIEWS = 8  # @param {type:"integer"}

# Equivale a la columna activo_referencia de tu Google Sheet: empareja
# una accion con el ETF contra el que debe medirse. Un nombre con
# referencia produce una view RELATIVA; el resto, ABSOLUTA.
REFERENCIAS = {
    'AAPL': 'QQQ', 'MSFT': 'QQQ', 'NVDA': 'QQQ', 'AVGO': 'SMH',
    'JPM': 'XLF', 'BAC': 'XLF', 'LLY': 'XLV', 'UNH': 'XLV',
    'XOM': 'XLE', 'CVX': 'XLE',
}

# Para lo que REFERENCIAS no cubre, el modelo busca contraparte entre los
# nombres de la cesta. Solo acepta el par si el spread es mas tranquilo
# que la pata suelta; si no, la view queda absoluta.
PARES_AUTOMATICOS = True  # @param {type:"boolean"}

# @markdown Cuántos nombres del ranking entran a la optimización.
TOP_N_CARTERA = 25  # @param {type:"integer"}
# @markdown Menos nombres = covarianza mejor estimada; más = más diversificación. Vive aquí y no en la celda de Cartera porque el pool de pares automáticos tiene que ser exactamente esta cesta.

from screener.optimizer import select_basket

_perfil_cci = profile_for_strategy(ESTRATEGIA_CCI)
if _perfil_cci.key != perfil.key:
    print(f'AVISO: corriste el screen con perfil {perfil.label} pero vas '
          f'a exportar para {ESTRATEGIA_CCI}, que corresponde a '
          f'{_perfil_cci.label}.')
    print('       Vuelve a la celda de Parametros y alinea ambos, o las '
          'views\n       llevaran umbrales y gates de otro mandato.')

_params = ViewParams(information_coefficient=IC_SUPUESTO,
                     max_views=MAX_VIEWS,
                     auto_pair=PARES_AUTOMATICOS)

# La cesta se arma antes que las views porque el pool de pares tiene que
# ser el universo de la covarianza: posterior() descarta en silencio
# cualquier view que nombre un ticker fuera de el, asi que un par contra
# un nombre que no llega a la cesta no debilita la view, la borra.
cartera_tickers = select_basket(scored, ESTRATEGIA_CCI,
                                top_n=TOP_N_CARTERA, min_per_class=3)

views = build_views(scored, market_data, strategy=ESTRATEGIA_CCI,
                    reference_map=REFERENCIAS,
                    pair_pool=cartera_tickers, params=_params)
cesta = build_basket(scored, strategy=ESTRATEGIA_CCI,
                     reference_map=REFERENCIAS)

print(f'{len(views)} views para {ESTRATEGIA_CCI} '
      f'(perfil {_perfil_cci.label}, IC {IC_SUPUESTO})\n')
_marca = {'declarado': ' (REFERENCIAS)', 'automatico': ' (par automático)'}
for _v in views:
    _quien = (_v['activo'] if _v['tipo'] == 'absoluto'
              else f"{_v['activo_long']} / {_v['activo_short']}")
    print(f"  {_v['tipo']:9s} {_quien:18s} Q {_v['Q']:+.2%}   "
          f"convicción {_v['conviccion']:.2f}"
          f"{_marca.get(_v.get('_pairing', ''), '')}")

_autom = [_v for _v in views if _v.get('_pairing') == 'automatico']
if _autom:
    print(f'\n{len(_autom)} par(es) los eligió el modelo, no REFERENCIAS. '
          f'Cada uno pasó el filtro de cobertura; el motivo va escrito '
          f'en la justificación de la view.')
elif PARES_AUTOMATICOS:
    print('\nNingún par automático: ningún candidato de la cesta cubría lo '
          'suficiente. Las views quedan absolutas, que es el resultado '
          'correcto cuando no hay con qué cubrir.')

# public_view quita la columna interna _q_bruto, que solo usa el
# diagnóstico de la celda siguiente y no viaja al archivo de CCI.
views_df = pd.DataFrame([public_view(_v) for _v in views])
cesta_df = pd.DataFrame(cesta)
views_df


## 10b · Diagnóstico del modelo

Dos mediciones sobre el modelo mismo, no sobre el mercado. Ninguna cambia una recomendación ni un peso: están para que sepas cuánto confiar en lo de arriba.

**Correlación entre bloques.** El modelo declara seis bloques y le asigna un peso a cada uno, lo que equivale a decir que cada bloque aporta información que los otros no tienen. Si dos bloques van juntos al 0.90, sus pesos son una sola apuesta hecha dos veces y la cartera está menos diversificada de lo que promete la tabla de pesos. El número de *factores efectivos* resume eso: si dice 2 sobre 6, tienes seis columnas midiendo dos cosas.

**Saturación de views.** La `Q` se recorta en ±5% porque así lo calibra el documento técnico de CCI. El recorte es una baranda; si casi todas las views terminan pegadas a ella, la baranda pasó a ser la señal: nombres que el screener rankeó muy distinto llegan al optimizador con el mismo retorno esperado y ese pedazo del ranking se tira a la basura. Dos views en el tope es normal; seis es un problema de calibración, y se arregla bajando el IC, no subiendo el tope.


In [ ]:
from screener.diagnostics import run_diagnostics

print(run_diagnostics(scored, views, _params))


## 11 · Cartera Black-Litterman

Aquí no hay archivo de por medio: `views` es una variable de Python que la celda anterior dejó en memoria, y esta la consume directo.

La covarianza usa contracción Ledoit-Wolf sobre retornos **diarios** — con ~52 barras semanales y más de 52 nombres la matriz sería singular — y la optimización respeta las bandas del Procedimiento de Inversión.

### El ancla: de dónde parte la cartera

`π = λ · Σ · w` es una multiplicación: la `w` que le pases **es** la cartera neutral. Con ocho views sobre veintitantos activos, esa `w` decide como tres cuartas partes del resultado. Es la decisión más grande de toda la asignación, y por eso el parámetro `ANCLA` está arriba del todo.

**`mercado`** era lo que hacía el sistema original: normalizar capitalización de acciones contra patrimonio de ETFs. Dos problemas. Primero, no son la misma unidad — la capitalización de una empresa es lo que vale la empresa; el patrimonio de un ETF es cuánta plata hay metida en ese envoltorio, y si el ETF es de renta variable está contando otra vez acciones que ya están en la cesta. Segundo, y peor: esa cuenta ancla cerca de 95% en renta variable. Ningún mandato de aquí permite eso. El resultado es que el optimizador se pasa el ejercicio empujando la cartera de vuelta contra el techo, y termina pegado exactamente en el límite — o sea, **la banda decide la asignación, no el modelo**.

**`politica`** (lo que corre por defecto) parte del **Modelo de Asignación de Mercado Internacional** de tu Procedimiento de Inversión: los porcentajes deseados por clase de activo, no una lectura de las bandas. Las bandas siguen siendo techos que se verifican; el Modelo es el objetivo. Dentro de cada línea del Modelo el reparto es por capitalización, con la banda de cada clase y el tope por nombre aplicados. La propiedad que importa: **sin views, el optimizador te devuelve exactamente esta cartera**. Las views se desvían de ahí, que es como debe funcionar.

| Clase | Cons. Def. | Conservador | Moderado | Agresivo |
|---|---|---|---|---|
| Renta fija gubernamental IG | 45% | 40% | 30% | 20% |
| Renta fija corporativa | 25% | 20% | 15% | 10% |
| Acciones y ETFs indexados | 20% | 30% | 50% | 65% |
| Efectivo / money market | 10% | 10% | 5% | 5% |

Dos cosas que conviene saber. **Materias primas no tienen línea en el Procedimiento**, así que el ancla no les asigna nada: el oro entra solo si una view lo empuja. Y si a alguna línea no le queda ninguna clase en la cesta, su porcentaje se reparte entre las demás al renormalizar, y la corrida lo dice.

### Tres arreglos frente al sistema original

1. **Solver.** Tu código pedía ECOS, que no viene en Colab; tu corrida guardada murió ahí sin producir cartera. Este usa CLARABEL, que viene con CVXPY.
2. **Apalancamiento.** `leverage_max` de 1.25 y 1.50 estaba declarado pero el optimizador fijaba `sum(w) == 1`, asi que nunca restringio nada. Ahora es un presupuesto real — pero **la mesa lo tiene apagado**: todas las carteras resuelven invertidas al 100%, sin importar lo que permita el mandato. El limite sigue en `REGULACIONES` porque es lo que dice el Procedimiento; la decision de no usarlo vive en `ALLOW_LEVERAGE`, en el optimizador.
3. **La auditoría ahora puede fallar.** `auditar_bandas` escribía "Auditoría OK" sin comparar nada. Esta compara contra cada límite y reporta lo que se rompe.

**Un aviso:** tus bandas no tienen clase para materias primas, y tu optimizador solo restringe las clases que aparecen en `bandas` — oro podía tomar el libro entero. Le puse un techo por perfil, pero **ese número lo inventé yo**, no sale de tu Procedimiento de Inversión. Confírmalo con Compliance antes de operar con esto.


In [ ]:
# @markdown Cartera neutral de la que parten las views.
ANCLA = "politica"  # @param ["politica", "mercado"]

# @markdown Posición mínima ejecutable, como fracción del libro.
POSICION_MINIMA = 0.01  # @param {type:"number"}
# @markdown El optimizador no sabe qué vale la pena operar: si le conviene, devuelve un 0.16% que cuesta una boleta, una línea en cada reporte y una conciliación para siempre. Las posiciones bajo este piso se eliminan **re-optimizando sin ellas**, no recortándolas del resultado — así las bandas del mandato siguen cumpliéndose exactas. Pon 0 para desactivarlo.

from screener.optimizer import (implied_equilibrium, market_weights,
                               optimize, policy_weights, posterior,
                               shrunk_covariance, allocation_table,
                               select_basket, gross_budget)
from screener.cci_regulation import REGULACIONES
from screener.cci_regulation import classify_for_bands
from screener.yahoo_adapter import daily_returns, fetch_market_caps

tipos_todos = {r.ticker: r.asset_type for r in scored}

# cartera_tickers viene de la celda de Views, que la necesita antes
# para acotar el pool de pares automaticos. Se recalcula aqui por si
# cambiaste TOP_N_CARTERA y corriste solo esta celda.
#
# La cesta no puede ser solo el top-N por score. El ranking premia
# momentum y riesgo-retorno, donde la renta variable domina, y con una
# cesta 100% equity el techo del mandato (60% en Moderado) queda por
# debajo del libro invertido: el solver responde 'infactible' y la
# cartera sale vacia. select_basket asegura representacion de cada
# clase disponible en el universo.
cartera_tickers = select_basket(scored, ESTRATEGIA_CCI,
                                top_n=TOP_N_CARTERA, min_per_class=3)

_clases_cesta = sorted({classify_for_bands(t, tipos_todos.get(t, 'ETF'))
                        for t in cartera_tickers})
print(f'Cesta: {len(cartera_tickers)} nombres en {len(_clases_cesta)} clases')
print(f'  {", ".join(_clases_cesta)}\n')

retornos = daily_returns(frame_diario, cartera_tickers)
covarianza = shrunk_covariance(retornos)

capitalizaciones = fetch_market_caps(list(covarianza.columns))
_tipos_cesta = {t: tipos_todos.get(t, 'ETF') for t in covarianza.columns}
# Presupuesto bruto en vigor. Con el apalancamiento apagado es 1.0.
_presupuesto = gross_budget(ESTRATEGIA_CCI)

if ANCLA == 'politica':
    pesos_ancla, _notas_ancla = policy_weights(
        _tipos_cesta, ESTRATEGIA_CCI, caps=capitalizaciones,
        total=_presupuesto)
    for _n in _notas_ancla:
        print(f'  {_n}')
else:
    pesos_ancla, sin_cap = market_weights(capitalizaciones,
                                          list(covarianza.columns))
    if sin_cap:
        print(f'Sin capitalizacion, excluidos del equilibrio: {sin_cap}')

print(f'\nAncla ({ANCLA}) por clase de activo:')
_cl_ancla = pd.Series({t: classify_for_bands(t, _tipos_cesta[t])
                       for t in pesos_ancla.index})
for _clase, _peso in pesos_ancla.groupby(_cl_ancla).sum().sort_values(
        ascending=False).items():
    if _peso > 0.0001:
        print(f'  {_peso:7.2%}  {_clase}')

pi = implied_equilibrium(pesos_ancla, covarianza)
er_posterior, cov_posterior = posterior(pi, covarianza, views)

tipos = tipos_todos
clases = {t: classify_for_bands(t, tipos.get(t, 'ETF'))
          for t in covarianza.columns}

cartera = optimize(er_posterior, cov_posterior, tipos, ESTRATEGIA_CCI,
                   min_position=POSICION_MINIMA or None)

print(f'{ESTRATEGIA_CCI}  |  estado: {cartera.status}')
print(f'Exposicion bruta   {cartera.gross_exposure:.1%}')
print(f'Retorno esperado   {cartera.expected_return:+.2%} anual')
print(f'Volatilidad        {cartera.volatility:.1%} anual')
print(f'Posiciones         {int((cartera.weights > 0).sum())}')

print('\nPor clase de activo')
for _clase, _peso in cartera.by_class.items():
    if _peso > 0.0001:
        print(f'  {_peso:7.2%}  {_clase}')

if cartera.breaches:
    print('\nAUDITORIA — INCUMPLIMIENTOS:')
    for _b in cartera.breaches:
        print(f'  {_b}')
else:
    print('\nAuditoria de bandas: sin incumplimientos.')
for _n in cartera.notes:
    print(f'NOTA: {_n}')

cartera_df = allocation_table(cartera, classes=clases)

if cartera_df.empty:
    # Una hoja vacia no dice nada. El motivo viaja con el resultado.
    cartera_df = pd.DataFrame({
        'ticker': ['SIN CARTERA'],
        'nombre': [f'La optimizacion no encontro solucion ({cartera.status})'],
        'clase_activo': [' | '.join(cartera.breaches) or 'sin detalle'],
        'peso': [0.0],
    })
    print('\nNO HAY CARTERA. Motivo:')
    for _b in cartera.breaches:
        print(f'  {_b}')

(cartera_df.style
    .format({'peso': '{:.2%}'})
    .map(lambda v: escala(v, 0, 0.12), subset=['peso'])
    .hide(axis='index')
    .set_caption(f'Cartera optimizada — {ESTRATEGIA_CCI}'))


## 12 · Descargar

**El Excel es para ti.** Ocho hojas: ranking, scores por bloque, comparación de perfiles, las views con su justificación, la cartera optimizada, la cesta, la cobertura de métricas y los parámetros de la corrida.

**El JSON es para tu sistema Black-Litterman**, no para leerlo. `black_litterman_core` hace `json.load()` y espera diccionarios de estructura heterogénea — una view absoluta trae `activo`, una relativa trae `activo_long` y `activo_short` — que en una tabla plana obligarían a celdas vacías. Y Excel coacciona tipos: una convicción de `0.85` puede volver como texto o mostrarse como 85%, y ese número entra directo en Ω. El nombre del archivo sigue la convención que tu propio `flujo_aprobacion` ya escribe en Drive.

Si no vas a alimentar el modelo BL hoy, desmarca la casilla y bájate solo el Excel.

### Dónde cae el archivo

En `CCI_BlackLitterman/propuestas/`, **nunca** en `aprobadas/`. Esa carpeta guarda las views que ya revisaste y justificaste, y tu `flujo_aprobacion` escribe ahí un archivo de la misma forma. Un archivo sin aprobar cayendo en esa ruta reemplazaría una decisión firmada por salida de máquina, sin dejar rastro. `write_views` se niega a escribir bajo `aprobadas/` aunque se lo pidas.

### Del lado de tu notebook BL

En el repo está `snippets/cci_bl_cargar_propuestas.py`: una celda para pegar entre `generar_propuestas_views` y `flujo_aprobacion`. Lee el archivo más reciente, avisa si está viejo, y fusiona con las propuestas de tu propio motor resolviendo duplicados por convicción — un mismo activo propuesto por ambas fuentes serían dos filas casi idénticas de P, lo que estrecha Ω artificialmente y le da a esa apuesta un peso que ninguna de las dos fuentes justifica sola.

El gestor sigue viendo cada view y decidiendo. Nada se aplica sin tu aprobación.


In [ ]:
EXPORTAR_JSON_PARA_BL = True  # @param {type:"boolean"}
# @markdown Desmárcalo si solo quieres el Excel.
GUARDAR_EN_DRIVE = False  # @param {type:"boolean"}
# @markdown Escribe las propuestas directo en `CCI_BlackLitterman/propuestas/` de tu Drive, para que el notebook BL las encuentre sin descargar ni subir nada.

from pathlib import Path

from screener.black_litterman import default_views_filename

ARCHIVO_EXCEL = 'screening.xlsx'
ARCHIVO_VIEWS = default_views_filename(ESTRATEGIA_CCI)

parametros = pd.DataFrame([
    ('Generado (UTC)', pd.Timestamp.utcnow().strftime('%Y-%m-%d %H:%M')),
    ('Perfil', perfil.label),
    ('Perfil — resumen', perfil.summary),
    ('Estrategia CCI destino', ESTRATEGIA_CCI),
    ('Universo', UNIVERSO),
    ('Nombres puntuados', len(scored)),
    ('Benchmark', BENCHMARK),
    ('Historia', PERIODO),
    ('Tasa libre de riesgo', f'{TASA_LIBRE_RIESGO:.2%}'),
    ('Posición asumida (liquidez)', f'${TAMANO_POSICION_USD:,.0f}'),
    ('Fuente de datos', market_data['data_source']),
    ('Portafolio', 'ninguno — screen independiente'),
    ('IC supuesto (views)', IC_SUPUESTO),
    ('Nota sobre el IC', 'supuesto declarado, no calibrado contra backtest'),
    ('Estado de la optimizacion', cartera.status),
    ('Exposicion bruta', f'{cartera.gross_exposure:.2%}'),
    ('Auditoria de bandas',
     'sin incumplimientos' if not cartera.breaches
     else ' | '.join(cartera.breaches)),
    ('Umbral Overweight', f'z >= {perfil.bands.overweight_z:+.2f}'),
    ('Umbral Underweight', f'z <= {perfil.bands.underweight_z:+.2f}'),
    ('Techo de volatilidad para OW',
     f'{perfil.gates.max_volatility_for_overweight:.0%}'),
    ('Beta máxima', f'{perfil.gates.beta_limit:.2f}'),
    ('Peso máximo por posición', f'{perfil.sizing.max_weight:.1%}'),
    ('Volumen diario mínimo', f'${perfil.eligibility.min_adv_usd/1e6:,.0f}MM'),
] + [(f'Peso — {b.label}', f'{b.weight:.0%}') for b in MODELO],
    columns=['Parámetro', 'Valor'])

# Las views en formato legible: una fila por view, con las dos formas
# (absoluta y relativa) resueltas a columnas explicitas.
views_excel = pd.DataFrame([{
    'tipo': v['tipo'],
    'activo': v.get('activo', ''),
    'long': v.get('activo_long', ''),
    'short': v.get('activo_short', ''),
    'Q': v['Q'],
    'conviccion': v['conviccion'],
    'justificacion': v['justificacion'],
} for v in views])

with pd.ExcelWriter(ARCHIVO_EXCEL, engine='openpyxl') as _xl:
    tabla.to_excel(_xl, sheet_name='Ranking', index=False)
    mapa.to_excel(_xl, sheet_name='Bloques')
    comparacion.to_excel(_xl, sheet_name='Perfiles')
    views_excel.to_excel(_xl, sheet_name='Views BL', index=False)
    cartera_df.to_excel(_xl, sheet_name='Cartera', index=False)
    cesta_df.to_excel(_xl, sheet_name='Cesta', index=False)
    universo.to_excel(_xl, sheet_name='Universo', index=False)
    _cov.to_excel(_xl, sheet_name='Cobertura', index=False)
    parametros.to_excel(_xl, sheet_name='Parametros', index=False)

    for _hoja in _xl.book.worksheets:
        _hoja.freeze_panes = 'A2'
        for _col in _hoja.columns:
            _ancho = max((len(str(c.value)) for c in _col if c.value), default=8)
            _hoja.column_dimensions[_col[0].column_letter].width = min(46, _ancho + 3)

print(f'{ARCHIVO_EXCEL}  —  {len(scored)} nombres, 8 hojas')

if EXPORTAR_JSON_PARA_BL:
    write_views(views, ARCHIVO_VIEWS, strategy=ESTRATEGIA_CCI,
                profile=_perfil_cci, meta=meta, params=_params)
    print(f'{ARCHIVO_VIEWS}  —  {len(views)} propuestas')

if EXPORTAR_JSON_PARA_BL and GUARDAR_EN_DRIVE:
    from screener.black_litterman import DRIVE_PROPOSALS_DIR
    from google.colab import drive
    drive.mount('/content/drive')
    _destino = (Path('/content/drive/MyDrive/CCI_BlackLitterman')
                / DRIVE_PROPOSALS_DIR / ARCHIVO_VIEWS)
    write_views(views, _destino, strategy=ESTRATEGIA_CCI,
                profile=_perfil_cci, meta=meta, params=_params)
    print(f'Guardado en Drive: {_destino}')

try:
    from google.colab import files
    files.download(ARCHIVO_EXCEL)
    if EXPORTAR_JSON_PARA_BL:
        files.download(ARCHIVO_VIEWS)
except ImportError:
    print('Fuera de Colab: los archivos quedaron en el directorio actual.')


## 13 · Ajuste fino del modelo

Los tres perfiles ya cubren la mayoría de los casos. Esto es para cuando quieras algo que ningún perfil expresa — mueve los pesos y vuelve a correr desde la celda 5, sin reiniciar el entorno.

**Ojo con el orden:** `run_standalone` vuelve a aplicar el perfil en cada llamada, así que sobrescribe lo que pongas aquí. Para que un ajuste manual sobreviva, usa `run(market_data, {}, standalone=True, target_position_usd=TAMANO_POSICION_USD)` en lugar de `run_standalone`.

`set_block_weights` acepta tamaños relativos y renormaliza. Un bloque en `0.0` se sigue calculando y mostrando, pero no aporta al compuesto: es la forma limpia de preguntar *¿qué dice el modelo sin momentum?*


In [ ]:
from screener.tuning import (block_weights, current_block_weights,
                             override, reset_all, set_block_weights)

# --- Ejemplo A: subir riesgo, bajar momentum ------------------------
# set_block_weights({'momentum': 0.10, 'risk': 0.25})

# --- Ejemplo B: quitar el techo de volatilidad para overweight ------
# override('GATES', max_volatility_for_overweight=None)

# --- Ejemplo C: bajar el minimo de liquidez a 5MM -------------------
# override('ELIGIBILITY', min_adv_usd=5_000_000)

# --- Ejemplo D: barrido de sensibilidad, sin efectos permanentes ----
# for _peso in (0.0, 0.11, 0.22, 0.44):
#     with block_weights({'momentum': _peso}):
#         _s, _ = run(market_data, {}, standalone=True,
#                     target_position_usd=TAMANO_POSICION_USD,
#                     rf=TASA_LIBRE_RIESGO)
#         _top = ', '.join(r.ticker for r in _s[:5])
#         print(f'momentum {_peso:.0%} -> {_top}')

# reset_all()   # vuelve a lo declarado en config.py

for _k, _w in current_block_weights().items():
    print(f'  {_w:6.1%}  {_k}')
